In [99]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from scipy.spatial.distance import pdist
from sklearn.preprocessing import OrdinalEncoder


1. EDA and Data Preparation

In [ ]:
# Data import
path = Path("Data") / "Dataset1_BankClients.xlsx" # general path

data = pd.read_excel(path)

In [ ]:
# Check if there are missing values
data.isna().sum()

# Data description
data.describe()

In [ ]:
data.info()
print()

In [ ]:
print(f'Numero duplicati: {data.duplicated().sum()}')
print('Valori mancanti per colonna:\n', data.isna().sum())

In [ ]:
# Scalers
mm_scaler = MinMaxScaler() # min-max scaler
z_scaler = StandardScaler() # z scaler

In [106]:
categorical_columns = ['Gender', 'Job', 'Area', 'CitySize', 'Investments']
numerical_columns = ['Income', 'Wealth', 'Debt', 'FinEdu', 'ESG', 'Digital', 'BankFriend', 'LifeStyle', 'Luxury', 'Saving']

# Split
numerical_features = data[numerical_columns]
categorical_features = data[categorical_columns].astype('category')

# Scale numerical
X_num = mm_scaler.fit_transform(numerical_features)

# Categorical representations
encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
X_cat_encoded = encoder.fit_transform(categorical_features)

X_cat_kproto = categorical_features.astype(str).to_numpy()

ord_encoder = OrdinalEncoder()
X_cat_int = ord_encoder.fit_transform(categorical_features)

# Combined datasets
X_kproto = np.hstack((X_num, X_cat_kproto))
X_encoded = np.hstack((X_num, X_cat_encoded))
X_int = np.hstack((X_num, X_cat_int))

In [ ]:
# Histograms of categorical variables
for col in categorical_columns:
    counts = data[col].value_counts()

    plt.figure()
    counts.plot(kind="bar")

    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")

    plt.xticks(rotation=45)
    plt.tight_layout()

    plt.show()

In [ ]:
# Histograms and distributions of numerical variables

for col in numerical_columns:
    sns.histplot(data[col].dropna(), bins=30, kde=True, color='skyblue')
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.grid()

    plt.tight_layout()
    plt.show()


In [ ]:
# Boxplots of numerical variabels
for col in numerical_columns:
    sns.boxplot(x=data[col], color='lightgreen')
    plt.title(f'Boxplot of {col}')
    plt.xlabel(col)
    plt.show()

In [ ]:
# Histograms, distributions and boxplots of numerical variables
for col in numerical_columns:
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram
    sns.histplot(data[col].dropna(), bins=30, kde=True, color='skyblue', ax=axes[0])
    axes[0].set_title(f"Distribution of {col}")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Frequency")
    axes[0].grid()

    # Boxplot
    sns.boxplot(x=data[col], color='lightgreen', ax=axes[1])
    axes[1].set_title(f"Boxplot of {col}")
    axes[1].set_xlabel(col)

    plt.tight_layout()
    plt.show()

In [ ]:
sns.pairplot(data[numerical_columns])

In [ ]:
for col in categorical_columns:
    print(data[col].value_counts(normalize=True).head())

In [ ]:
distances = pdist(data[numerical_columns], metric="euclidean")

sns.histplot(distances, bins=50)
plt.title("Distribuzione delle distanze")

In [ ]:
# Correlation matrix
corr = numerical_features.corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Numerical variables correlation matrix")
plt.show()

In [ ]:
abs_corr = corr.abs()
upper = abs_corr.where(np.triu(np.ones(abs_corr.shape), k=1).astype(bool))
result = upper.stack().sort_values(ascending=False)
print(result)

2. Distances


2.1. K-Prototypes

$d(x,y) = \sum (x_i-y_i)^2 + \gamma \sum \delta(x_j,y_j)$,\
where $\delta=1$ if $x_j {=}\mathllap{/} y_j$ and $\delta=0$ if $x_j = y_j$

numericals -> distanza euclidea \
categoricals -> matching

In [96]:
from kmodes.kprototypes import KPrototypes

kproto = KPrototypes(
    n_clusters=3, # k=3 gives the best silhouette score
    init="Cao",
    random_state=42
)

cat_indices = list(range(len(numerical_columns),
                         len(numerical_columns) + len(categorical_columns)))

clusters = kproto.fit_predict(
    Y_mm,
    categorical=cat_indices
)

In [ ]:
data["cluster"] = clusters
data.groupby("cluster")[numerical_columns].mean()

In [ ]:
for col in categorical_columns:
    print(data.groupby("cluster")[col].value_counts(normalize=True))

In [ ]:
def kproto_distance(x_num, x_cat, y_num, y_cat, gamma=1.0):
    num_dist = np.sum((x_num - y_num) ** 2)
    cat_dist = np.sum(x_cat != y_cat)
    return num_dist + gamma * cat_dist

def compute_kproto_distance_matrix(X_num, X_cat, gamma=1.0):
    n = X_num.shape[0]
    D = np.zeros((n, n))

    for i in range(n):
        for j in range(i + 1, n):
            d = kproto_distance(X_num[i], X_cat[i], X_num[j], X_cat[j], gamma)
            D[i, j] = d
            D[j, i] = d

    return D

from sklearn.metrics import silhouette_score

X_cat_kproto = categorical_features.astype(str).to_numpy()

D = compute_kproto_distance_matrix(X_num_mm, X_cat_kproto, gamma=1.0)

score = silhouette_score(D, clusters, metric="precomputed")
print("Silhouette score:", score)

In [ ]:
from sklearn.metrics import davies_bouldin_score

db_score = davies_bouldin_score(X_mm,clusters)
print(db_score)

2.2. Personalized Mixed Distance

$d(x,y) = \alpha d_{num} + (1-\alpha)d_{cat}$

In [ ]:
categorical_codes = categorical_features.apply(lambda col: col.cat.codes).to_numpy()

In [ ]:
# Hamming distance for categorical variabels

def hamming_dstance(point_1, point_2):
    """ 
    Compute the Hamming distance: the Hamming distance between two strings or vectors of equal length is the number of positions at which the corresponding symbols are different.
    In this case it is modified for integers: return 1 if integers are different, otherwise return 0.
    """
    distance = np.sum(point_1 != point_2)
    return distance

In [ ]:
from scipy.spatial.distance import pdist, squareform

Z = categorical_features.to_numpy()
D_hamming = squareform(pdist(Z, metric="hamming"))
D_hamming_counts = D_hamming * Z.shape[1]

In [ ]:
# Tanimoto distance
from scipy.spatial.distance import rogerstanimoto

m = X_cat.shape[0]
D_tanimoto = np.zeros((m,m))

for i in range(m):
    for j in range(m):
        D_tanimoto[i,j] = rogerstanimoto(X_cat[i], X_cat[j])